<img height="100" src="logotmg.png" width="900" alt="Talia Morchi" id="tmg_logo" title="Data Analyst Course">
<center><h1><u>Welcome to the python for data analysis course </u></h1></center><center><h2>Class 6</h2></center>

## Install pyodbc

#### The pyodbc library must be installed once, and then we will only import it into the workbook
* Start >> Anaconda prompt >> write the following command, and press Enter:
       pip install pyodbc

## Import Basic libraries

In [1]:
# If pyodbc is not installed, please install it first using pip install pyodbc
import pandas as pd
import pyodbc

## Connection to SQL

#### Define the connection to MSSQL

In [2]:
# Server = SQL server name
# Database = The name of the db we want to connect to

conn = pyodbc.connect ( 'Driver={SQL Server};'
                        'Server=localhost\\SQLEXPRESS;'
                        'Database=AdventureWorks2019;'
                        'Trusted_Connection=yes;' )

## Retrive data from MSSQL table
sql_query_result = pd.read_sql_query('SQL Query' ,conn)

#### Basic queries

In [3]:
# Read SQL query results into a DataFrame
# Simple example
query = '''SELECT * 
           FROM Sales.SalesOrderHeader'''

sql_query_result = pd.read_sql_query(query, conn)

In [4]:
# Receive results into DataFrame object
print (type (sql_query_result))

<class 'pandas.core.frame.DataFrame'>


In [5]:
sql_query_result.head(3)

,SalesOrderID,RevisionNumber,OrderDate,DueDate,ShipDate,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,...,CreditCardID,CreditCardApprovalCode,CurrencyRateID,SubTotal,TaxAmt,Freight,TotalDue,Comment,rowguid,ModifiedDate
0,43659,8,2011-05-31,2011-06-12,2011-06-07,5,False,SO43659,PO522145787,10-4020-000676,...,16281.0,105041Vi84182,NaN,20565.6206,1971.5149,616.0984,23153.2339,None,79B65321-39CA-4115-9CBA-8FE0903E12E6,2011-06-07
1,43660,8,2011-05-31,2011-06-12,2011-06-07,5,False,SO43660,PO18850127500,10-4020-000117,...,5618.0,115213Vi29411,NaN,1294.2529,124.2483,38.8276,1457.3288,None,738DC42D-D03B-48A1-9822-F95A67EA7389,2011-06-07
2,43661,8,2011-05-31,2011-06-12,2011-06-07,5,False,SO43661,PO18473189620,10-4020-000442,...,1346.0,85274Vi6854,4.0,32726.4786,3153.7696,985.5530,36865.8012,None,D91B9131-18A4-4A11-BC3A-90B6F53E9D74,2011-06-07


In [6]:
# Triple quotes preserve the text structure
query = '''SELECT BusinessEntityID,
                  FirstName,
                  LastName,
                  ModifiedDate
           FROM   Person.Person
           WHERE  BusinessEntityID = 6 '''

pd.read_sql_query(query, conn)

,BusinessEntityID,FirstName,LastName,ModifiedDate
0,6,Jason,Globe,2020-11-12 10:57:49.890


In [7]:
# Example with more complex query
query = '''SELECT   h.SalesOrderID,
                    h.OrderDate,
                    d.ProductID,
                    d.OrderQty,
                    d.LineTotal
           FROM Sales.SalesOrderHeader h
               JOIN Sales.SalesOrderDetail d
                   ON h.SalesOrderID = d.SalesOrderID
           WHERE YEAR(h.OrderDate) = 2013'''

pd.read_sql_query(query ,conn)

,SalesOrderID,OrderDate,ProductID,OrderQty,LineTotal
0,49181,2013-01-01,794,1,2181.5625
1,49182,2013-01-01,790,1,2443.3500
2,49183,2013-01-01,791,1,2443.3500
3,49184,2013-01-01,784,1,2049.0982
4,49185,2013-01-01,784,1,2049.0982
...,...,...,...,...,...
56568,63360,2013-12-31,998,1,539.9900
56569,63360,2013-12-31,872,1,8.9900
56570,63361,2013-12-31,955,1,2384.0700
56571,63362,2013-12-31,966,1,2384.0700


## Cursor object
The cursor object that gives us the ability to have multiple seperate working environments through the same connection to the database. By default the cursor is created using the default cursor class.

#### Queries with parameters

In order to retrieve results of query with parameters, we'll follow the next steps:
1.  cursor = conn.cursor()
2.  query = 'Query syntax'
3.  cursor.execute (query)
4.  if just recsive results >>   results = cursor.fetchall() 
5.  if changes the data >>  conn.commit()
    
    
Another way is to concatenate text to create a regular text query

In [8]:
# Create cursor object that is bound to the connection 
cursor = conn.cursor()

#### Change sql data

In [9]:
# The question marks (?), will be replaced by the values of a parameter that will be sent

query = '''UPDATE Person.Person
           SET    FirstName = ?, LastName = ?, ModifiedDate=getdate()
           WHERE  BusinessEntityID = ? ;'''

In [10]:
# Execute the query and send the parameters to replace the '?'

cursor.execute (query,['Jason','Globe',6])
# cursor.execute (query,['Jossef','Goldberg',6])

In [11]:
# Commit the changes in the SQL DB
# We will use commit() only if there are changes in the data
conn.commit()

In [12]:
# Check if the value changed

query = '''SELECT BusinessEntityID,
                  FirstName,
                  LastName,
                  ModifiedDate
           FROM   Person.Person
           WHERE  BusinessEntityID = 6 '''

pd.read_sql_query(query, conn)

,BusinessEntityID,FirstName,LastName,ModifiedDate
0,6,Jason,Globe,2020-11-12 12:13:23.620


#### Retrive data using query with parameters

In [13]:
# Displays the distance of the sale from the annual sales average, only for existing customers in 2013
query = '''SELECT SalesOrderID,
                  TotalDue,
                  TotalDue - ( SELECT avg (TotalDue)
                               FROM Sales.SalesOrderHeader h
                               WHERE year (OrderDate) = ?) DistFromAvg
           FROM   sales.SalesOrderHeader
           WHERE  year (OrderDate) = ? and
                  CustomerID in ( SELECT distinct CustomerID
                                  FROM   Sales.SalesOrderHeader
                                  WHERE  year (OrderDate) < ?
                                      );'''


In [14]:
year = 2013

In [15]:
# Prepare and execute a database query or command
cursor.execute (query,[year, year, year])

In [16]:
# Fetch all remaining rows of a query result
# We use fetchall when we just retreive data (do not update)
rows = cursor.fetchall()

In [17]:
# The data received in a list of rows object
print(type(rows))
print(type(rows[0]))

<class 'list'>
<class 'pyodbc.Row'>


In [18]:
rows

[(49443, Decimal('76.2853'), Decimal('-3376.3933')),
 (49444, Decimal('7333.3076'), Decimal('3880.6290')),
 (49445, Decimal('3173.9282'), Decimal('-278.7504')),
 (49446, Decimal('15340.6533'), Decimal('11887.9747')),
 (49447, Decimal('120.2717'), Decimal('-3332.4069')),
 (49448, Decimal('1435.1443'), Decimal('-2017.5343')),
 (49449, Decimal('50.7737'), Decimal('-3401.9049')),
 (49450, Decimal('36854.3018'), Decimal('33401.6232')),
 (49451, Decimal('2921.0624'), Decimal('-531.6162')),
 (49452, Decimal('1473.8636'), Decimal('-1978.8150')),
 (49453, Decimal('1399.4512'), Decimal('-2053.2274')),
 (49454, Decimal('6972.6272'), Decimal('3519.9486')),
 (49455, Decimal('528.9880'), Decimal('-2923.6906')),
 (49456, Decimal('700.6543'), Decimal('-2752.0243')),
 (49457, Decimal('76.2966'), Decimal('-3376.3820')),
 (49458, Decimal('223.6817'), Decimal('-3228.9969')),
 (49459, Decimal('732.9386'), Decimal('-2719.7400')),
 (49460, Decimal('2464.5599'), Decimal('-988.1187')),
 (49461, Decimal('203.09

#### Convert row object into DataFrame object

In [21]:
# Short syntax of for loop
# cursor.description >> It is a sequence of Column instances, each one describing one result column in order.
# Create a list of all column names

[col[0] for col in cursor.description]

# col_names = [col[0] for col in cursor.description]

In [22]:
# Convert structured or record ndarray to DataFrame

df = pd.DataFrame.from_records(rows, columns=col_names)
df

,SalesOrderID,TotalDue,DistFromAvg
0,49443,76.2853,-3376.3933
1,49444,7333.3076,3880.6290
2,49445,3173.9282,-278.7504
3,49446,15340.6533,11887.9747
4,49447,120.2717,-3332.4069
...,...,...,...
3360,63351,1379.9572,-2072.7214
3361,63352,2710.6092,-742.0694
3362,63353,2728.3003,-724.3783
3363,63361,2634.3974,-818.2812


In [23]:
# Close all connections
cursor.close()
conn.close()